# Silver Layer: Sales Details CDC (Free Edition)

**Pattern**: CDC via `foreachBatch` Merge  
**Trigger**: AvailableNow (Incremental Batch)

In [0]:
from pyspark.sql.functions import col
from delta.tables import DeltaTable

bronze_table = "workspace.bronze.crm_sales_details_cdc"
silver_table = "workspace.silver.crm_sales_details_cdc"
checkpoint_path = "/Volumes/workspace/bronze/raw_sources/checkpoints/silver_crm_sales_details"

def upsert_sales(batch_df, batch_id):
    transformed_df = batch_df.select(
        col("sls_id").alias("sales_id"),
        col("sls_ord_id").alias("order_id"),
        col("sls_prd_id").alias("product_id"),
        col("sls_cust_id").alias("customer_id"),
        col("sls_ord_dt").alias("order_date"),
        col("sls_ship_dt").alias("ship_date"),
        col("sls_due_dt").alias("due_date"),
        col("sls_sales").alias("sales_amount"),
        col("sls_qty").alias("quantity"),
        col("ingestion_timestamp")
    ).filter(col("sales_id").isNotNull())

    if not spark.catalog.tableExists(silver_table):
        transformed_df.write.format("delta").saveAsTable(silver_table)
        return

    target_table = DeltaTable.forName(spark, silver_table)
    (target_table.alias("t")
        .merge(transformed_df.alias("s"), "t.sales_id = s.sales_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())

print(f"Starting incremental update for {silver_table}...")
query = (
    spark.readStream
    .table(bronze_table)
    .writeStream
    .foreachBatch(upsert_sales)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print(f"✓ Silver update complete. Total records in {silver_table}: {spark.table(silver_table).count():,}")